In [ ]:
import os
import json
import gc
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

def run_vllm_inference(model_path, output_dir, output_file, tensor_parallel_size=1):
    """
    Hàm chạy inference tự động cho bất kỳ model nào.
    """
    print(f"\n{'='*60}")
    print(f"🚀 BẮT ĐẦU CHẠY MODEL: {model_path}")
    print(f"📂 Thư mục lưu: {output_dir}/{output_file}")
    print(f"{'='*60}")
    
    os.makedirs(output_dir, exist_ok=True)

    # ==========================================
    # 1. CHUẨN BỊ DỮ LIỆU & PROMPT
    # ==========================================
    print("⏳ Loading tokenizer and dataset...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    data = load_dataset('VoCuc/MetaMathQA-50k-256', split='train')['query']

    print("🧩 Applying chat template...")
    prompts_all = []
    for prompt in data:
        conversation = [
            {"role": "system", "content": "You are a teacher. Solve the problem and put your final answer within \\boxed{}."},
            {"role": "user", "content": prompt}
        ]
        formatted_prompt = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False
        )
        prompts_all.append(formatted_prompt)

    # ==========================================
    # 2. KHỞI TẠO vLLM ENGINE
    # ==========================================
    print("🔥 Initializing vLLM...")
    llm = LLM(
        model=model_path,
        trust_remote_code=True,
        dtype="bfloat16",
        tensor_parallel_size=tensor_parallel_size,
        gpu_memory_utilization=0.9,
        max_seq_len_to_capture=1024,
        seed=42
    )

    # ==========================================
    # 3. CHẠY INFERENCE
    # ==========================================
    sampling_params = SamplingParams(
        temperature=0.85,
        top_p=0.95,
        max_tokens=512,
        skip_special_tokens=True
    )

    print(f"⚙️ Generating responses for {len(prompts_all)} prompts...")
    outputs = llm.generate(prompts_all, sampling_params)

    # ==========================================
    # 4. LƯU KẾT QUẢ
    # ==========================================
    output_data = []
    for i, output in enumerate(outputs):
        generated_text = output.outputs[0].text.strip()
        output_data.append({
            'prompt': data[i],
            'generated_text': generated_text,
        })

    output_path = os.path.join(output_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in output_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Đã lưu xong file tại: {output_path}")

    print("🧹 Đang dọn dẹp bộ nhớ GPU...")
    # Phá hủy đối tượng LLM để PyTorch biết vùng nhớ này không còn dùng nữa
    del llm
    del tokenizer
    # Gọi Garbage Collector của Python
    gc.collect()
    # Ép CUDA dọn dẹp các cache đang chiếm giữ
    torch.cuda.empty_cache()
    print("✨ Sẵn sàng cho model tiếp theo!\n")



In [ ]:
models_to_test = [
    {
        "model_path": "Qwen/Qwen2.5-Math-1.5B",
        "output_dir": "data/dpo/Qwen/Qwen2.5-Math-1.5B",
        "output_file": "generated_train.jsonl"
    },
    # {
    #     "model_path": "Qwen/Qwen2.5-0.5B",
    #     "output_dir": "data/dpo/Qwen/Qwen2.5-0.5B",
    #     "output_file": "generated_train.jsonl"
    # }
]

for config in models_to_test:
    run_vllm_inference(
        model_path=config["model_path"],
        output_dir=config["output_dir"],
        output_file=config["output_file"],
        tensor_parallel_size=1
    )